# Surrogate Factory Notebook template  
## Chapter 2. Data Generation and Adquisition

Objectives of this template:
- Define and build the Design of Experiments (DoE)
- Compute dependencies and restrictions defined in the excel template
- Launch Sim tool to get outputs values
- Execute ETL to build the DataSet

### 0.- Define general config of the workflow

Surrogate Factory Initialization

In [ ]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()


_Note: if error occurs, check if the environment and kernel are properly configured. Check [Surrogate Factory User Manual](https://docs.google.com/document/d/1VEYCL0I4XwuFt3_tCZ3j4Fm-Ow8ubVapGlakIQWd77w/edit?usp=sharing)_ 

### 3. Data Adquisition/Generation

#### Create DoE

Let's first generate the Design Of Experiments. This is defined in the Create_Design_of_Experiments where you can define independance variables, dependencies and some other conditions

In [ ]:
workflow.import_metadata(stage_name="SF_2_Data_Acquisition_Generation")
JSON(workflow.metadata.to_json())

#### 3.1 Create Basic DoE

In [ ]:
from data_generation.define_doe import define_doe
define_doe(workflow)

In [ ]:
from data_generation.create_inputs import create_inputs
output_table_1 = create_inputs(workflow)
#output_table_1.describe()

#### 3.2 Scale

In [ ]:
#from data_acquisition_generation.data_generation.create_design_of_experiment import scale
from data_generation.create_inputs import scale
output_table_2 = scale(workflow, output_table_1)
#output_table_2.describe()

#### 3.3 Extend

In [ ]:
#from data_acquisition_generation.data_generation.create_design_of_experiment import extend
from data_generation.create_inputs import extend
output_table_3 = extend(workflow, output_table_2)
#output_table_3.describe()

#### 3.4 Filter

In [ ]:
#from data_acquisition_generation.data_generation.create_design_of_experiment import filter
from data_generation.create_inputs import filter
input_table_1 = filter(workflow, output_table_3)
#input_table_1.describe()

### 3.5 Generate data

In [ ]:
#from data_acquisition_generation.data_generation.generate_data import isami_batch
from data_generation.get_outputs import launch_sim
# this step can be skipped if you have already generated the files  
# if not, uncomment the next line to execute full process  

launch_sim(workflow, input_table_1)

#### 3.6 Results Parser

#### 3.6.1 Extract

In [ ]:
from data_acquisition.outputs_parser import batch_extract
output_object_1 = batch_extract(workflow, input_table_1)
output_object_1

#### 3.6.2 Transform

In [ ]:
from data_acquisition.outputs_parser import batch_transform
output_object_2 = batch_transform(workflow, output_object_1)
output_object_2

#### 3.6.3 Load

In [ ]:
from data_acquisition.outputs_parser import batch_load
Dataset = batch_load(workflow, output_object_2 , input_table_1)
Dataset

## Visualization and Checks

In [ ]:
%matplotlib inline
from data_acquisition.outputs_parser import data_visualization
data_visualization(workflow, Dataset)

## Save Data

In [ ]:


outputFilename = workflow.config['job_name'] + '_raw_dataset.csv'
workflow.save_data(Dataset,  outputFilename)

In [ ]:
workflow.save_metadata()